# Worksheet 2.2: Hot and Cold LLMs

Today we'll explore a key setting that controls how AI generates text - and actually *see* the probabilities that AIs use when writing.

---
## Part A: What Does Temperature Do? (Discovery)

In the opening activity, we built sentences by voting on the next word. But how does the AI decide which words to offer - and how does it choose between them?

Let's experiment with a setting called `temperature` and see if we can figure out what it does.

### Setup

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

### Write Your Prompts

Each group member writes their OWN prompts - do not copy from your neighbor!

In [ ]:
# Prompt A: Something with ONE correct answer
# Examples: "The capital of Japan is", "The year World War 2 ended was"
my_factual_prompt = "___"

# Prompt B: Something creative - ask for 2-3 sentences
# Example: "Write 2-3 sentences about a robot who learns to dance"
my_creative_prompt = "___"

### Experiment Round 1

Run each cell **3 times**. Pay attention to the outputs.

In [ ]:
# Run this cell 3 times
response = client.models.generate_content(
    model='gemma-3n-e2b-it',
    contents=my_factual_prompt,
    config={"temperature": 0.0}
)
print(response.text)

In [ ]:
# Run this cell 3 times
response = client.models.generate_content(
    model='gemma-3n-e2b-it',
    contents=my_creative_prompt,
    config={"temperature": 0.0}
)
print(response.text)

Quick check with your neighbor: What do you notice?

### Experiment Round 2

Same prompts. Run each cell **3 times** again.

In [ ]:
# Run this cell 3 times
response = client.models.generate_content(
    model='gemma-3n-e2b-it',
    contents=my_factual_prompt,
    config={"temperature": 1.5}
)
print(response.text)

In [ ]:
# Run this cell 3 times
response = client.models.generate_content(
    model='gemma-3n-e2b-it',
    contents=my_creative_prompt,
    config={"temperature": 1.5}
)
print(response.text)

### Group Discussion

**Exercise A.1** Compare your results with your group members:

- What changed between Round 1 and Round 2?
- What do you think `temperature` controls?

**Write your group's hypothesis here:**

*Your answer*

---
## Part B: Looking Inside the AI's Brain

You have a hypothesis about temperature. Now let's look inside the AI to see what's actually happening.

We're using GPT-2, an older (2019) but fully open model that lets us see the probabilities it uses.

### Setup

In [ ]:
#@title Run this cell first (takes about 30 seconds)

!pip install transformers --quiet

import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import warnings
warnings.filterwarnings('ignore')

print("Loading GPT-2... ", end="")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Done!")

# ============================================================
# FUNCTION 1: Show next word probabilities
# ============================================================
def show_next_word_probabilities(text, top_k=10):
    """Show what the AI thinks are the most likely next words."""
    inputs = tokenizer.encode(text, return_tensors='pt')

    with torch.no_grad():
        outputs = model(inputs)

    logits = outputs.logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, top_k)

    print(f'Prompt: "{text}"')
    print(f"\nTop {top_k} predictions for the next word:\n")

    for i in range(top_k):
        word = tokenizer.decode(top_indices[i].item())
        prob = top_probs[i].item() * 100
        bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
        print(f"  {word:15} {bar} {prob:5.1f}%")

# ============================================================
# FUNCTION 2: Show temperature effect with "other" category
# ============================================================
def show_temperature_effect(text, temperatures=[0.5, 1.0, 2.0], top_k=5):
    """Show how temperature changes the probability distribution."""
    inputs = tokenizer.encode(text, return_tensors='pt')

    with torch.no_grad():
        outputs = model(inputs)
    logits = outputs.logits[0, -1, :]

    print(f'Prompt: "{text}"\n')

    for temp in temperatures:
        adjusted = logits / temp
        probs = F.softmax(adjusted, dim=-1)

        top_probs, top_indices = torch.topk(probs, top_k)
        top_sum = top_probs.sum().item() * 100
        other_prob = 100 - top_sum

        if temp < 1:
            label = f"Temperature {temp}"
        elif temp == 1:
            label = f"Temperature {temp}"
        else:
            label = f"Temperature {temp}"

        print("=" * 55)
        print(label)
        print("=" * 55)

        for i in range(top_k):
            word = tokenizer.decode(top_indices[i].item())
            prob = top_probs[i].item() * 100
            bar = "*" * int(prob / 2) + "." * (50 - int(prob / 2))
            print(f"  {word:12} {bar} {prob:5.1f}%")

        # Show "other" category
        other_bar = "*" * int(other_prob / 2) + "." * (50 - int(other_prob / 2))
        print(f"  {'[other]':12} {other_bar} {other_prob:5.1f}%  <- {50257 - top_k:,} other words")
        print()

# ============================================================
# FUNCTION 3: Generate step by step
# ============================================================
def generate_step_by_step(prompt, num_words=8, temperature=1.0):
    """Generate text and show what was chosen at each step."""
    input_ids = tokenizer.encode(prompt, return_tensors='pt')

    print(f'Starting prompt: "{prompt}"')
    print(f"Temperature: {temperature}\n")
    print("Step-by-step generation:")
    print("=" * 60)

    for step in range(num_words):
        with torch.no_grad():
            outputs = model(input_ids)

        logits = outputs.logits[0, -1, :]
        adjusted = logits / temperature
        probs = F.softmax(adjusted, dim=-1)

        # Sample from the distribution
        next_token = torch.multinomial(probs, num_samples=1)
        chosen_word = tokenizer.decode(next_token.item())
        chosen_prob = probs[next_token.item()].item() * 100

        # Get top 3 options
        top_probs, top_indices = torch.topk(probs, 3)
        top_tokens = top_indices.tolist()

        # Check if chosen word is in top 3
        chosen_in_top3 = next_token.item() in top_tokens

        print(f"\nStep {step + 1}: Top 3 options were:")
        for i in range(3):
            word = tokenizer.decode(top_indices[i].item())
            prob = top_probs[i].item() * 100
            if top_indices[i].item() == next_token.item():
                print(f"  '{word}': {prob:.1f}% <- CHOSEN")
            else:
                print(f"  '{word}': {prob:.1f}%")

        # If chosen word was NOT in top 3, show it separately
        if not chosen_in_top3:
            print(f"     ...")
            print(f"  (many other words...)")
            print(f"     ...")
            print(f"  '{chosen_word}': {chosen_prob:.2f}% <- CHOSEN (surprise pick!)")

        # Add to sequence
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    final_text = tokenizer.decode(input_ids[0])
    print("\n" + "=" * 60)
    print(f"Final result: \"{final_text}\"")

# ============================================================
print("\n" + "="*50)
print("GPT-2 is ready! Functions available:")
print("  - show_next_word_probabilities(text)")
print("  - show_temperature_effect(text)")
print("  - generate_step_by_step(prompt, temperature=1.0)")
print("="*50)

### What Does the AI Think Comes Next?

When you give the AI some text, it calculates a **probability for every possible next word** (all 50,257 of them!).

Let's see what GPT-2 thinks should come after "The weather today is":

In [ ]:
show_next_word_probabilities("The weather today is")

**Exercise B.1:** What will happen when you run this? Write your prediction first, then run the cell.

In [ ]:
show_next_word_probabilities(Once upon a time)

*Your prediction:*

*What actually happened:*

**Exercise B.2:** Fill in the blank with a prompt about animals.

In [ ]:
show_next_word_probabilities(___)

---
## Part C: How Temperature Changes the Distribution

Now let's see how temperature reshapes the probabilities.

In [ ]:
show_temperature_effect("The best thing about Lunar New Year is that", temperatures=[0.5, 1.0, 2.0])

**Exercise C.1:** Look at the percentages AND the "[other]" category. What happens as temperature increases?

- The TOP word's probability:
- The "[other]" category (all 50,000+ remaining words):

*Your answer*

**Exercise C.2:** What words do you think will have the highest probability? Write your prediction, then run the cell.

In [ ]:
show_next_word_probabilities("The capital of Vietnam is")

*Your prediction:*

*What actually happened (was it what you expected?):*

**Exercise C.3:** Fill in the blanks to compare a very cold and very hot temperature.

In [ ]:
show_temperature_effect("The secret to happiness is", temperatures=[___, ___])

---
## Part D: How Does Sampling Work?

The AI doesn't just pick the highest probability word. It **samples** from the distribution - exactly like our opening activity.

Let's watch the AI generate text step by step:

In [ ]:
generate_step_by_step("The best thing about Lunar New Year is", temperature=1.0)

**Exercise D.1:** Run the cell above multiple times. Does the AI always choose the highest probability word? Did you see any "surprise picks"?

*Your answer*

Now try with different temperatures:

In [ ]:
# Low temperature - more predictable
generate_step_by_step("The best thing about Lunar New Year is", temperature=0.3)

In [ ]:
# High temperature - more random, more "surprise picks"
generate_step_by_step("The best thing about Lunar New Year is", temperature=2.0)

**Exercise D.2:** Compare the three outputs (temp 0.3, 1.0, 2.0). How many "surprise picks" (words outside top 3) did you see at each temperature?

*Your answer*

**Exercise D.3:** What will happen? Write your prediction first.

In [ ]:
generate_step_by_step("The robot walked into the", temperature=0.1)

In [ ]:
generate_step_by_step("The robot walked into the", temperature=1.9)

*Your prediction:*

*What actually happened:*

**Exercise D.4:** This code runs without errors. But is the output what you expect? Predict what the AI will generate, then run it.

In [ ]:
generate_step_by_step("The President of Vietnam is", temperature=0.001, num_words=5)

*Your prediction:*

*What actually happened (why might this be?):*

---
## Part E: Your Experiments

One handy Python tip for generating your own code is that you can get additional information about a function by just running the function with an question mark after it.  Look at the help information below (these functions are defined in the setup cell you ran above, so they won't work in a brand new notebook):

In [ ]:
show_next_word_probabilities?

In [ ]:
show_temperature_effect?

In [ ]:
generate_step_by_step?

**Exercise E.1:** Look at the examples above. Write your own cell that generates 12 words starting with a prompt of your choice, using temperature 0.8.  Show the probabilities at each step.

In [ ]:
# Your code here


**Exercise E.2:** For your own prompt, show the predictions for the next word at temperatures, 0.2, .9, and 1.75.

In [ ]:
# Your code here

---
## Part F: The Big Picture

### What we learned:

1. **LLMs don't "know" one answer** - they have probabilities over ALL possible next words (50,257 of them!)

2. **Temperature reshapes the distribution:**
   - Low temp: sharp distribution (top word dominates, "other" is tiny)
   - High temp: flat distribution (probability spreads to "other" words)

3. **Sampling adds randomness** - even with the same probabilities, different runs give different results. High temperature = more "surprise picks"

4. **This is why LLMs can be creative** - they don't just pick the "best" word, they explore the possibility space

**Exercise F.1:** Write a cell that shows the top 15 most likely next words for a prompt of your choice. Hint: look at the help for `show_next_word_probabilities.

In [ ]:
# Your code here


**Exercise F.2:** How does this connect to the "Shoggoth" idea from Class 1.2? What does temperature reveal about the nature of LLMs?

*Your answer*

---
## Part G: Explore the Distribution

Let's visualize exactly how temperature transforms the probability distribution.

In [ ]:
#@title Run this cell to set up the interactive visualization

!pip install ipywidgets --quiet

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# Store logits globally for the visualization
_viz_logits = None
_viz_prompt = None

def create_temperature_visualization(prompt):
    """Create an interactive visualization of temperature's effect on the distribution."""
    global _viz_logits, _viz_prompt

    # Get logits for this prompt
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = model(inputs)
    _viz_logits = outputs.logits[0, -1, :]
    _viz_prompt = prompt

    # Create output area
    output = widgets.Output()

    # Create slider
    temp_slider = widgets.FloatSlider(
        value=1.0,
        min=0.1,
        max=3.0,
        step=0.1,
        description='Temperature:',
        continuous_update=True,
        readout=True,
        readout_format='.1f',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='500px')
    )

    def update_visualization(temperature):
        with output:
            clear_output(wait=True)

            # Apply temperature and get probabilities
            adjusted = _viz_logits / temperature
            probs = F.softmax(adjusted, dim=-1)

            # Get top 100 for the chart
            top_probs, top_indices = torch.topk(probs, 100)
            top_probs_np = top_probs.numpy()

            # Get top 5 for display
            top5_probs, top5_indices = torch.topk(probs, 5)

            # Calculate statistics
            top1_prob = top_probs[0].item() * 100
            top5_prob = top5_probs.sum().item() * 100

            # Calculate words needed for 90%
            sorted_probs, _ = torch.sort(probs, descending=True)
            cumsum = torch.cumsum(sorted_probs, dim=0)
            words_for_90 = (cumsum < 0.9).sum().item() + 1

            # Create figure with two columns
            fig = plt.figure(figsize=(12, 5))

            # Left side: bar chart
            ax1 = fig.add_axes([0.05, 0.15, 0.55, 0.75])
            bars = ax1.bar(range(1, 101), top_probs_np * 100, color='steelblue', width=0.8)
            ax1.set_xlabel('Token Rank')
            ax1.set_ylabel('Probability (%)')
            ax1.set_title(f'Top 100 Token Probabilities (Temperature = {temperature:.1f})')
            ax1.set_xlim(0, 101)
            ax1.set_ylim(0, max(top_probs_np * 100) * 1.1 + 1)

            # Right side: statistics and top words
            ax2 = fig.add_axes([0.68, 0.15, 0.30, 0.75])
            ax2.axis('off')

            # Statistics text
            stats_text = f"""STATISTICS

Top word probability: {top1_prob:.1f}%

Top 5 combined: {top5_prob:.1f}%

Words needed for 90%: {words_for_90:,}


TOP 5 WORDS
"""

            for i in range(5):
                word = tokenizer.decode(top5_indices[i].item())
                prob = top5_probs[i].item() * 100
                # Clean up the word display
                word_display = repr(word)[1:-1]  # Show spaces clearly
                stats_text += f"\n{i+1}. {word_display}: {prob:.1f}%"

            ax2.text(0.0, 0.95, stats_text, transform=ax2.transAxes,
                    fontsize=11, verticalalignment='top', fontfamily='monospace')

            plt.suptitle(f'Prompt: "{_viz_prompt}"', fontsize=12, y=0.98)
            plt.show()

    # Connect slider to update function
    temp_slider.observe(lambda change: update_visualization(change['new']), names='value')

    # Display widgets
    display(temp_slider)
    display(output)

    # Initial display
    update_visualization(1.0)

print("Interactive visualization ready!")
print("Use: create_temperature_visualization(\"your prompt here\")")

In [ ]:
#@title Enter a prompt to explore, then press "play"
visualization_prompt = "I did great on the Mini-Test because I drank a lot of " #@param {type:"string"}

create_temperature_visualization(visualization_prompt)

### Reflection Questions

**G.1:** Set temperature to 0.2. How much probability does the top word have? How many words to reach 90%?

*Your answer*

**G.2:** Set temperature to 2.0. Answer the same questions. What changed?

*Your answer*

**G.3:** Now keep temperature at 1.0. Can you find a prompt where the distribution is already "spread out" (top word < 20%)? Can you find one where it's very "focused" (top word > 50%)? Compare with your group.

*Your prompts and observations*

**G.4:** What makes some prompts more spread out than others, even at the same temperature?

*Your answer*

---
## Submission

1. Make sure you've completed all exercises and challenges
2. Share this notebook with **ethan.brown@fulbright.edu.vn** as an **Editor**
3. Submit the link on Canvas

# Credits

This notebook contains materials created by Ethan C. Brown, collaborating with Claude (see [conversation 1](https://claude.ai/share/0a0838b1-d283-489c-a1b1-7d975b6a1b53), [conversation 2](https://claude.ai/share/4a57eac2-d618-4942-aaff-6964fe09c00a)), and heavily inspired by Tufino, E. (2025). Exploring large language models (LLMs) through interactive Python activities. *Physics Education, 60*(5), 055003. https://doi.org/10.1088/1361-6552/adea28
 .